# 作业 04：正则表达式与结构化网页数据

| 项目 | 内容 |
| --- | --- |
| 学号 | 20244080130 |
| 姓名 | 廖晨宇 |
| 作业编号 | 作业 4 |
| Conda 环境 | `exp03`（独立虚拟环境，非 base） |
| Python 版本 | 3.12 |
| 提交文件 | `20244080130_廖晨宇_hw04.ipynb` |
| 运行方式 | 重启内核 → Run All，按顺序执行全部单元格 |

> 说明：每道编程题都给出了函数实现 + 多组测试用例（含边界输入），测试用 `assert` 校验并把结果打印出来。

## 一、作业环境与提交说明

1. 使用独立 conda 环境 **`exp03`**（Python 3.12），未使用 base 环境；环境名由第一个代码单元打印的 `CONDA_DEFAULT_ENV` 验证。
2. 第一个代码单元按要求填写学号，并记录机器指纹与 `RUN_ID`。
3. 第二部分为 12 道正则表达式练习，每题一个 Markdown 说明 + 一个代码单元（函数 + 测试用例）。
4. 第三部分从维基百科城市页面抓取气候数据表格，实现 `retrieve_climate_table()` 与 `list_climate_table_row_names()` 两个函数，并用多个城市页面（含结构差异页面）测试。
5. 提交前重启内核并 Run All，全部代码单元按顺序执行、无报错。

In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080130
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] exp03
[学号]       20244080130
[机器指纹] 01534316381bc475
[RUN_ID]   20244080130-01534316381bc475


## 二、Python 正则表达式练习

下面 12 道题都写成函数，并用多组测试用例验证：先打印输入与输出，再用 `assert` 断言预期值，断言不通过会直接抛 `AssertionError`。

### 1. 检查字符串是否仅包含英文字母 a-z、A-Z 和数字 0-9

用 `re.fullmatch(r"[A-Za-z0-9]+", s)`：`fullmatch` 要求**整个字符串**都匹配，字符类只允许这 62 个字符；`+` 要求至少 1 个字符，所以空串返回 False。

In [2]:
import re


def only_alnum(s: str) -> bool:
    # 字符串是否只由英文字母和数字组成（空串视为 False）
    return re.fullmatch(r"[A-Za-z0-9]+", s) is not None


cases = ["Python3", "abc", "123", "abc 123", "abc-123", "中文abc", "", "a_b", "ABC789xyz"]
for s in cases:
    print(f"{s!r:12} -> {only_alnum(s)}")

assert only_alnum("Python3") is True
assert only_alnum("abc-123") is False
assert only_alnum("中文abc") is False
assert only_alnum("") is False
print("测试通过")

'Python3'    -> True
'abc'        -> True
'123'        -> True
'abc 123'    -> False
'abc-123'    -> False
'中文abc'      -> False
''           -> False
'a_b'        -> False
'ABC789xyz'  -> True
测试通过


### 2. 匹配一个 a 后面跟着零个或多个 b

`re.fullmatch(r"ab*", s)`：`*` 表示 `b` 出现 0 次或多次，因此 `"a"` 与 `"abbb"` 都匹配。

In [3]:
def a_zero_or_more_b(s: str) -> bool:
    return re.fullmatch(r"ab*", s) is not None


for s in ["a", "ab", "abb", "abbb", "b", "abbc", "", "aa"]:
    print(f"{s!r:8} -> {a_zero_or_more_b(s)}")

assert a_zero_or_more_b("a") is True
assert a_zero_or_more_b("abbb") is True
assert a_zero_or_more_b("b") is False
assert a_zero_or_more_b("aa") is False
print("测试通过")

'a'      -> True
'ab'     -> True
'abb'    -> True
'abbb'   -> True
'b'      -> False
'abbc'   -> False
''       -> False
'aa'     -> False
测试通过


### 3. 匹配一个 a 后面跟着一个或多个 b

`re.fullmatch(r"ab+", s)`：`+` 要求 `b` 至少出现 1 次，所以 `"a"` 不匹配。

In [4]:
def a_one_or_more_b(s: str) -> bool:
    return re.fullmatch(r"ab+", s) is not None


for s in ["a", "ab", "abb", "abbb", "b", "ac"]:
    print(f"{s!r:8} -> {a_one_or_more_b(s)}")

assert a_one_or_more_b("ab") is True
assert a_one_or_more_b("a") is False
assert a_one_or_more_b("b") is False
print("测试通过")

'a'      -> False
'ab'     -> True
'abb'    -> True
'abbb'   -> True
'b'      -> False
'ac'     -> False
测试通过


### 4. 匹配以 a 开头、以 b 结尾，中间可以有任意字符

`re.fullmatch(r"a.*b", s)`：`.*` 匹配中间的任意字符（不跨行），首尾分别锚定 `a` 和 `b`。`"ab"` 中间为空也算。

In [5]:
def a_start_b_end(s: str) -> bool:
    return re.fullmatch(r"a.*b", s) is not None


for s in ["ab", "axb", "axxxb", "a", "b", "ba", "abx", "a1b2b"]:
    print(f"{s!r:8} -> {a_start_b_end(s)}")

assert a_start_b_end("ab") is True
assert a_start_b_end("axxxb") is True
assert a_start_b_end("a") is False
assert a_start_b_end("ba") is False
print("测试通过")

'ab'     -> True
'axb'    -> True
'axxxb'  -> True
'a'      -> False
'b'      -> False
'ba'     -> False
'abx'    -> False
'a1b2b'  -> True
测试通过


### 5. 删除 IP 地址各段数字的前导零

`re.sub(r"\b0+(?=\d)", "", ip)`：

- `\b` 保证只从每一段的开头动手；
- `0+` 吞掉一段前导零；
- `(?=\d)` 是**前瞻**，要求零后面还必须跟着数字，这样整段全是 0 时（如 `000`）会留下最后一个 `0`，而 `0.0.0.0` 不会被破坏。

In [6]:
def strip_ip_leading_zeros(ip: str) -> str:
    return re.sub(r"\b0+(?=\d)", "", ip)


tests = {
    "192.168.001.010": "192.168.1.10",
    "010.000.000.001": "10.0.0.1",
    "0.0.0.0": "0.0.0.0",
    "192.168.100.020": "192.168.100.20",
    "255.255.255.255": "255.255.255.255",
}
for src, expect in tests.items():
    got = strip_ip_leading_zeros(src)
    print(f"{src:20} -> {got:20} 期望 {expect}")
    assert got == expect, (src, got, expect)
print("测试通过")

192.168.001.010      -> 192.168.1.10         期望 192.168.1.10
010.000.000.001      -> 10.0.0.1             期望 10.0.0.1
0.0.0.0              -> 0.0.0.0              期望 0.0.0.0
192.168.100.020      -> 192.168.100.20       期望 192.168.100.20
255.255.255.255      -> 255.255.255.255      期望 255.255.255.255
测试通过


### 6. 将空格替换为下划线，并将下划线替换为空格

不能连续调用两次 `re.sub`（第二遍会把刚生成的下划线又换回去），所以用一次替换 + 回调函数：命中空白就产出 `_`，命中下划线就产出空格。

In [7]:
def swap_space_underscore(s: str) -> str:
    return re.sub(r"[ _]", lambda m: "_" if m.group() == " " else " ", s)


s = "user name_test  a_b"
got = swap_space_underscore(s)
print("原文：", repr(s))
print("交换：", repr(got))

assert swap_space_underscore("a b") == "a_b"
assert swap_space_underscore("a_b") == "a b"
assert swap_space_underscore("") == ""
assert got == "user_name test__a b"
print("测试通过")

原文： 'user name_test  a_b'
交换： 'user_name test__a b'
测试通过


### 7. 将 yyyy-mm-dd 转换为 dd-mm-yyyy

用三个捕获组取出年、月、日，替换串按 `\3-\2-\1` 重排。`re.sub` 默认替换所有出现，一行里有多个日期也能全换。

In [8]:
def to_dd_mm_yyyy(s: str) -> str:
    return re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3-\2-\1", s)


print(to_dd_mm_yyyy("2024-05-01"))
print(to_dd_mm_yyyy("截止 2024-12-31，开始 2025-01-01"))

assert to_dd_mm_yyyy("2024-05-01") == "01-05-2024"
assert to_dd_mm_yyyy("2024-12-31 与 2025-01-01") == "31-12-2024 与 01-01-2025"
assert to_dd_mm_yyyy("没有日期") == "没有日期"
print("测试通过")

01-05-2024
截止 31-12-2024，开始 01-01-2025
测试通过


### 8. 找出所有以 a 或 e 开头的单词

`re.findall(r"\b[aeAE]\w*", s)`：`\b` 定位单词开头，`[aeAE]` 限定首字母，`\w*` 吞掉单词剩余部分。正则默认区分大小写，所以显式把大写 `A`、`E` 也放进字符类。

In [9]:
def words_starting_with_a_or_e(s: str) -> list:
    return re.findall(r"\b[aeAE]\w*", s)


s = "an apple, an egg and an orange. Elephant eats every apple. Banana is not one."
found = words_starting_with_a_or_e(s)
print("句子：", s)
print("结果：", found)

assert found[0] == "an"
assert "apple" in found and "egg" in found and "Elephant" in found
assert "Banana" not in found
print("测试通过")

句子： an apple, an egg and an orange. Elephant eats every apple. Banana is not one.
结果： ['an', 'apple', 'an', 'egg', 'and', 'an', 'Elephant', 'eats', 'every', 'apple']
测试通过


### 9. 将 Road 缩写为 Rd

`re.sub(r"\bRoad\b", "Rd", s)`：两侧的 `\b` 保证只替换独立的 `Road`，像 `Broadway` 这样的词内部不会误伤。

In [10]:
def abbreviate_road(s: str) -> str:
    return re.sub(r"\bRoad\b", "Rd", s)


print(abbreviate_road("221B Baker Road, Broadway Street"))
print(abbreviate_road("Roads are Road-like"))

assert abbreviate_road("Main Road") == "Main Rd"
assert abbreviate_road("Broadway") == "Broadway"
assert abbreviate_road("Road") == "Rd"
print("测试通过")

221B Baker Rd, Broadway Street
Roads are Rd-like
测试通过


### 10. 找出长度至少 4 个字符的单词

`re.findall(r"\b\w{4,}\b", s)`：`{4,}` 表示 4 个及以上，`\b` 保证整词匹配（不会把 `cat` 从 `catsup` 里抠出来）。

In [11]:
def long_words(s: str) -> list:
    return re.findall(r"\b\w{4,}\b", s)


s = "The quick brown fox jumps over the lazy dog"
print("句子：", s)
print("长度 ≥ 4 的单词：", long_words(s))

assert long_words(s) == ["quick", "brown", "jumps", "over", "lazy"]
assert long_words("a ab abc abcd") == ["abcd"]
assert long_words("") == []
print("测试通过")

句子： The quick brown fox jumps over the lazy dog
长度 ≥ 4 的单词： ['quick', 'brown', 'jumps', 'over', 'lazy']
测试通过


### 11. 删除字符串中除字母和数字之外的所有字符

`re.sub(r"[^A-Za-z0-9]", "", s)`：字符类取反集合，把不允许的字符替换成空串。

In [12]:
def keep_alnum_only(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]", "", s)


for s in ["ABC-123 xyz_789!", "a b\tc\n\td", "北京市海淀区 100084", "!!!" ]:
    print(f"{s!r:26} -> {keep_alnum_only(s)!r}")

assert keep_alnum_only("ABC-123 xyz_789!") == "ABC123xyz789"
assert keep_alnum_only("!!!") == ""
print("测试通过")

'ABC-123 xyz_789!'         -> 'ABC123xyz789'
'a b\tc\n\td'              -> 'abcd'
'北京市海淀区 100084'            -> '100084'
'!!!'                      -> ''
测试通过


### 12. 删除括号及括号内的内容

逐行处理 `example (.com)`、`w3resource`、`github (.com)`、`stackoverflow (.com)`，预期依次得到 `example`、`w3resource`、`github`、`stackoverflow`。

`re.sub(r"\s*\([^)]*\)", "", line)`：`\s*` 连括号前的空格一起去掉，`\([^)]*\)` 匹配一对括号及其内部，用 `[^)]*` 而不是 `.*` 可以避免跨过多个括号匹配。

In [13]:
def remove_parentheses(s: str) -> str:
    return re.sub(r"\s*\([^)]*\)", "", s)


lines = [
    "example (.com)",
    "w3resource",
    "github (.com)",
    "stackoverflow (.com)",
]
results = [remove_parentheses(x) for x in lines]
for src, out in zip(lines, results):
    print(f"{src:26} -> {out!r}")

assert results == ["example", "w3resource", "github", "stackoverflow"]
assert remove_parentheses("a (b) c (d) e") == "a c e"
assert remove_parentheses("no parens here") == "no parens here"
print("测试通过")

example (.com)             -> 'example'
w3resource                 -> 'w3resource'
github (.com)              -> 'github'
stackoverflow (.com)       -> 'stackoverflow'
测试通过


## 三、从网页检索气候数据

维基百科的城市页面里，气候数据放在标题以 **“Climate data for …”** 开头的 `wikitable` 表格中。本部分用 `requests` 取 HTML、`BeautifulSoup(..., "html.parser")` 解析，实现两个函数：

- `retrieve_climate_table(s)`：有气候数据表则返回 `Tag` 对象，读取成功但没找到返回 `None`，网络/HTTP 错误抛异常；
- `list_climate_table_row_names(s)`：返回该表的行名列表（纯字符串），找不到返回 `None`。

**实际使用的 URL 与访问日期**：见下一个代码单元打印的清单（本机运行时为 2026-11-26）。

> 网络说明：本机直连 `en.wikipedia.org` 会被重置，因此运行时通过环境变量 `HTTPS_PROXY` 指定本地代理；未设置该变量时代码走直连。这也算“网络错误应抛出异常”的一种情形，下面的测试用例会演示 404 页面抛出的 `HTTPError`。

In [14]:
import datetime
import re
import socket
import time

import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (edu homework, python-requests)"}
PROXY = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
print("运行时间：", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("代理设置：", PROXY or "未设置（直连）")

_cache = {}


def fetch_html(url: str, timeout: int = 40, attempts: int = 3) -> str:
    # 下载页面并返回解码后的 HTML；网络错误与 HTTP 错误都以异常形式抛出。
    # 连接类错误（超时、连接重置、TLS 中断）会重试 attempts 次，仍失败则抛出最后一次异常；
    # HTTP 4xx/5xx 不重试，直接抛 requests.HTTPError。
    if url in _cache:
        return _cache[url]
    session = requests.Session()
    session.headers.update(HEADERS)
    if PROXY:
        session.proxies.update({"http": PROXY, "https": PROXY})

    last_error = None
    for i in range(1, attempts + 1):
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()              # 4xx / 5xx 抛 requests.HTTPError
            if not resp.encoding or resp.encoding.lower() in ("iso-8859-1", "ascii"):
                resp.encoding = resp.apparent_encoding or "utf-8"
            _cache[url] = resp.text
            return resp.text
        except requests.HTTPError:
            raise                                # 状态码错误不重试
        except (requests.ConnectionError, requests.Timeout) as e:
            last_error = e
            print(f"  第 {i} 次请求失败（{type(e).__name__}），{1.5 * i:.1f}s 后重试…")
            time.sleep(1.5 * i)
    raise last_error


def collapse(text: str) -> str:
    # 把连续空白字符合并成一个空格并去掉首尾空白
    return re.sub(r"\s+", " ", text).strip()


CLIMATE_RE = re.compile(r"climate\s+data\s+for", re.I)
NAV_PREFIX_RE = re.compile(r"^(v\s*t\s*e|v\s*·\s*d\s*·\s*e)\s+", re.I)


def table_climate_title(table) -> str | None:
    # 若表格是气候数据表，返回其标题文本，否则返回 None。
    # 标题可能出现在 <caption>，也可能位于第一行（常见为 colspan 的 <th>）；
    # Los Angeles / Hyderabad 这类页面的标题前面还带有 "v t e" 导航文字，需要先剥掉。
    chunks = []
    caption = table.find("caption")
    if caption is not None:
        chunks.append(collapse(caption.get_text(" ")))
    body = table.find("tbody") or table
    first_row = body.find("tr")
    if first_row is not None:
        chunks.append(collapse(first_row.get_text(" ")))
    for chunk in chunks:
        cleaned = NAV_PREFIX_RE.sub("", chunk)
        if CLIMATE_RE.search(cleaned):
            return cleaned
    return None


# 城市页面清单（含结构差异页面），后面几个代码单元都会用到
CITY_URLS = {
    "Harbin": "https://en.wikipedia.org/wiki/Harbin",
    "Madison, Wisconsin": "https://en.wikipedia.org/wiki/Madison,_Wisconsin",
    "Buenos Aires": "https://en.wikipedia.org/wiki/Buenos_Aires",
    "Los Angeles": "https://en.wikipedia.org/wiki/Los_Angeles",
    "Hyderabad": "https://en.wikipedia.org/wiki/Hyderabad",
    "Boston": "https://en.wikipedia.org/wiki/Boston",
}
print(f"计划测试的城市页面：{len(CITY_URLS)} 个")
for name, url in CITY_URLS.items():
    print(f"  {name:20} {url}")

运行时间： 2026-11-26 16:36:32
代理设置： http://127.0.0.1:7897
计划测试的城市页面：6 个
  Harbin               https://en.wikipedia.org/wiki/Harbin
  Madison, Wisconsin   https://en.wikipedia.org/wiki/Madison,_Wisconsin
  Buenos Aires         https://en.wikipedia.org/wiki/Buenos_Aires
  Los Angeles          https://en.wikipedia.org/wiki/Los_Angeles
  Hyderabad            https://en.wikipedia.org/wiki/Hyderabad
  Boston               https://en.wikipedia.org/wiki/Boston


### 任务 1：读取页面、定位气候数据表格并观察结构

先取 **Harbin** 页面：统计页面里 `<table>` 总数，找出标题匹配 “Climate data for” 的表格，然后观察它的直接子节点数量与标签名、行数、标题行与月份表头。

In [15]:
url = CITY_URLS["Harbin"]
html = fetch_html(url)
soup = BeautifulSoup(html, "html.parser")

print(f"URL：{url}")
print(f"HTML 字符数：{len(html)}")
tables = soup.find_all("table")
print(f"页面 <table> 总数：{len(tables)}")

hits = [(i, t) for i, t in enumerate(tables) if table_climate_title(t)]
print(f"其中气候数据表格：{len(hits)} 个，位置索引 {[i for i, _ in hits]}")

idx, table = hits[0]
children = [c for c in table.children if getattr(c, "name", None)]
print("\n== 表格结构 ==")
print("class 属性：", table.get("class"))
print("直接子节点数：", len(children), "| 标签名：", [c.name for c in children])

tbody = table.find("tbody")
rows = tbody.find_all("tr", recursive=False)
print("行数（tbody 下 <tr>）：", len(rows))

title_cells = rows[0].find_all(["th", "td"], recursive=False)
print("标题行：cell 数 =", len(title_cells),
      "| 标签 =", [c.name for c in title_cells],
      "| colspan =", [c.get("colspan") for c in title_cells])
print("标题文本：", collapse(title_cells[0].get_text(" ")))

month_cells = rows[1].find_all(["th", "td"], recursive=False)
print("月份表头：cell 数 =", len(month_cells),
      "| 标签 =", [c.name for c in month_cells[:3]], "...",
      "| 文本示例 =", [collapse(c.get_text(" ")) for c in month_cells[:3]])

print("\n前 3 个数据行的第一个单元格（行名）：")
for tr in rows[2:5]:
    first = tr.find_all(["th", "td"], recursive=False)[0]
    print(f"  {first.name} scope={first.get('scope')!r} 文本={collapse(first.get_text(' '))!r}")

print("\n最后一行（数据来源）：")
last = rows[-1].find_all(["th", "td"], recursive=False)
print(" ", last[0].name, "colspan=", last[0].get("colspan"), collapse(last[0].get_text(" "))[:60])

URL：https://en.wikipedia.org/wiki/Harbin
HTML 字符数：1273035
页面 <table> 总数：40
其中气候数据表格：2 个，位置索引 [3, 4]

== 表格结构 ==
class 属性： ['wikitable', 'mw-collapsible']
直接子节点数： 1 | 标签名： ['tbody']
行数（tbody 下 <tr>）： 18
标题行：cell 数 = 1 | 标签 = ['th'] | colspan = ['14']
标题文本： Climate data for Harbin, elevation 118 m (387 ft), (1991–2020 normals, extremes 1951–present)
月份表头：cell 数 = 14 | 标签 = ['th', 'th', 'th'] ... | 文本示例 = ['Month', 'Jan', 'Feb']

前 3 个数据行的第一个单元格（行名）：
  th scope='row' 文本='Record high °C (°F)'
  th scope='row' 文本='Mean maximum °C (°F)'
  th scope='row' 文本='Mean daily maximum °C (°F)'

最后一行（数据来源）：
  td colspan= 14 Source 2: China Meteorological Administration, [ 91 ] [ 92 ]


**结构说明（任务 1）**：Harbin 页面共有 40 个 `<table>`，其中 2 个是气候数据表。以第一个为例：

- 表格只有一个直接子节点 `<tbody>`，行都在 `tbody` 里——所以取行时要用 `tbody.find_all("tr", recursive=False)`，而不是直接在 `<table>` 上找；
- 第 1 行是标题行：**一个** `colspan="14"` 的 `<th>`，文本以 `Climate data for Harbin …` 开头；
- 第 2 行是月份表头：第一格 `th scope="row"` 文本为 `Month`，后面 13 格是 `th scope="col"` 的 Jan…Dec 与 Year；
- 之后的数据行都是「`th scope="row"` 行名 + 13 个 `<td>` 数值」；
- 最后一行是数据来源，第一格是 `td colspan="14"`（Boston 页面这里则是 `th colspan="14"`，这也是后面判断行名时要同时检查「首格不是 `th`」和「首格 `colspan` 大于 1」的原因）。

### 任务 2：`retrieve_climate_table(s)`

函数只收一个 URL：取页面 → 遍历所有 `<table>` → 用 `table_climate_title()` 判断标题（caption 或第一行）是否含 “Climate data for” → 命中就返回该 `Tag`；读到了页面但没有匹配表格返回 `None`；网络/HTTP 错误由 `fetch_html` 抛出。

测试用例覆盖 6 个城市（含 Los Angeles、Hyderabad 这种标题带 `v t e` 前缀的结构差异页面），另外加两个边界用例：无气候表的普通页面（返回 `None`）、不存在的页面（抛 `HTTPError`）。

In [16]:
def retrieve_climate_table(s: str):
    # 返回页面的气候数据表格（BeautifulSoup Tag）；无此表格返回 None；网络/HTTP 错误抛异常。
    html = fetch_html(s)
    soup = BeautifulSoup(html, "html.parser")
    for table in soup.find_all("table"):
        if table_climate_title(table):
            return table
    return None


results = {}
for name, url in CITY_URLS.items():
    table = retrieve_climate_table(url)
    results[name] = table
    if table is None:
        print(f"{name:20} -> None")
    else:
        title = table_climate_title(table)
        n_rows = len((table.find("tbody") or table).find_all("tr", recursive=False))
        print(f"{name:20} -> <table {table.get('class')}> 行数={n_rows}")
        print(f"{'':20}    标题：{title[:70]}")

# 边界用例 1：页面可访问，但没有气候数据表格
no_climate_url = "https://en.wikipedia.org/wiki/Banana"
print(f"\n无气候表页面 {no_climate_url} ->", retrieve_climate_table(no_climate_url))

# 边界用例 2：页面不存在，应抛 HTTPError
bad_url = "https://en.wikipedia.org/wiki/ThisPageDoesNotExist_20244080130"
try:
    retrieve_climate_table(bad_url)
    print("没有抛异常（不符合预期）")
except requests.HTTPError as e:
    print(f"不存在的页面抛出了 {type(e).__name__}：", e)

assert all(v is not None for v in results.values()), "六个城市页面都应找到气候数据表"
assert retrieve_climate_table(no_climate_url) is None
print("\n测试通过")

Harbin               -> <table ['wikitable', 'mw-collapsible']> 行数=18
                        标题：Climate data for Harbin, elevation 118 m (387 ft), (1991–2020 normals,


Madison, Wisconsin   -> <table ['wikitable', 'mw-collapsible']> 行数=18
                        标题：Climate data for Madison, Wisconsin ( Dane County Regional Airport ), 


Buenos Aires         -> <table ['wikitable', 'mw-collapsible']> 行数=14
                        标题：Climate data for Buenos Aires Central Observatory, located in Agronomí


Los Angeles          -> <table ['wikitable', 'mw-collapsible']> 行数=16
                        标题：Climate data for Los Angeles ( Dodger Stadium , Downtown ), 1991–2020 


Hyderabad            -> <table ['wikitable', 'mw-collapsible']> 行数=17
                        标题：Climate data for Hyderabad (1991–2020 normals, extremes 1951–2020)


Boston               -> <table ['wikitable', 'mw-collapsible']> 行数=20
                        标题：Climate data for Boston, Massachusetts ( Logan Airport ), 1991−2020 no



无气候表页面 https://en.wikipedia.org/wiki/Banana -> None


不存在的页面抛出了 HTTPError： 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/ThisPageDoesNotExist_20244080130



测试通过


**测试结果说明（任务 2）**：6 个城市页面都返回了 `<table>` 对象；`Banana` 页面能正常读取但返回 `None`；不存在的页面由 `raise_for_status()` 抛出 `requests.HTTPError`（404），符合“网络或 HTTP 错误应抛出异常”的要求。

Los Angeles 与 Hyderabad 的标题前面带有 `v t e` 导航文字，直接判断“表格文本是否以 Climate data for 开头”会漏掉它们——所以 `table_climate_title()` 先用 `NAV_PREFIX_RE` 剥掉前缀再做匹配。

### 任务 3：`list_climate_table_row_names(s)`

对气候表的每一行做判断：

1. 跳过没有单元格的行；
2. **首格必须是 `<th>`**（行名）；首格是 `<td>` 的行（如数据来源行）跳过；
3. **整行都是 `<th>`** 的跳过（月份表头行、标题行）；
4. 首格 `colspan > 1` 的跳过（标题行、Boston 那种 `<th colspan="14">Source…</th>`）；
5. 文本以 “Climate data for” 开头的跳过（防止标题行漏网）；
6. 其余取首格文本，合并连续空白并去除首尾空白后加入列表。

这样既不依赖 `scope="row"` 属性（Boston 页面没有它），也能避开 “Climate data for…” 标题行、月份列标题与数据来源行。

In [17]:
def list_climate_table_row_names(s: str):
    # 返回气候数据表的行名列表；成功读取但无气候表返回 None。
    table = retrieve_climate_table(s)
    if table is None:
        return None

    body = table.find("tbody") or table
    names = []
    for tr in body.find_all("tr", recursive=False):
        cells = tr.find_all(["th", "td"], recursive=False)
        if not cells:
            continue
        first = cells[0]
        if first.name != "th":                            # 1) 行名必须在首格，且是 th
            continue
        if all(c.name == "th" for c in cells):            # 2) 纯表头行（月份列标题）
            continue
        colspan = first.get("colspan")
        if colspan and int(colspan) > 1:                  # 3) 跨整行的标题/来源
            continue
        text = collapse(first.get_text(" "))
        if not text or CLIMATE_RE.search(text):           # 4) 标题行兜底
            continue
        names.append(text)
    return names


for name, url in CITY_URLS.items():
    names = list_climate_table_row_names(url)
    print(f"\n{name}（{len(names)} 项）：")
    for i, n in enumerate(names, 1):
        print(f"  {i:2}. {n}")


Harbin（14 项）：
   1. Record high °C (°F)
   2. Mean maximum °C (°F)
   3. Mean daily maximum °C (°F)
   4. Daily mean °C (°F)
   5. Mean daily minimum °C (°F)
   6. Mean minimum °C (°F)
   7. Record low °C (°F)
   8. Average precipitation mm (inches)
   9. Average precipitation days (≥ 0.1 mm)
  10. Average snowy days
  11. Average relative humidity (%)
  12. Mean monthly sunshine hours
  13. Percentage possible sunshine
  14. Average ultraviolet index



Madison, Wisconsin（15 项）：
   1. Record high °F (°C)
   2. Mean maximum °F (°C)
   3. Mean daily maximum °F (°C)
   4. Daily mean °F (°C)
   5. Mean daily minimum °F (°C)
   6. Mean minimum °F (°C)
   7. Record low °F (°C)
   8. Average precipitation inches (mm)
   9. Average snowfall inches (cm)
  10. Average extreme snow depth inches (cm)
  11. Average precipitation days (≥ 0.01 in)
  12. Average snowy days (≥ 0.1 in)
  13. Average relative humidity (%)
  14. Mean monthly sunshine hours
  15. Percentage possible sunshine



Buenos Aires（11 项）：
   1. Record high °C (°F)
   2. Mean daily maximum °C (°F)
   3. Daily mean °C (°F)
   4. Mean daily minimum °C (°F)
   5. Record low °C (°F)
   6. Average precipitation mm (inches)
   7. Average precipitation days (≥ 0.1 mm)
   8. Average relative humidity (%)
   9. Mean monthly sunshine hours
  10. Mean daily sunshine hours
  11. Average ultraviolet index



Los Angeles（12 项）：
   1. Record high °F (°C)
   2. Mean maximum °F (°C)
   3. Mean daily maximum °F (°C)
   4. Daily mean °F (°C)
   5. Mean daily minimum °F (°C)
   6. Mean minimum °F (°C)
   7. Record low °F (°C)
   8. Average precipitation inches (mm)
   9. Average precipitation days (≥ 0.01 in)
  10. Mean monthly sunshine hours
  11. Percentage possible sunshine
  12. Average ultraviolet index



Hyderabad（13 项）：
   1. Record high °C (°F)
   2. Mean daily maximum °C (°F)
   3. Daily mean °C (°F)
   4. Mean daily minimum °C (°F)
   5. Record low °C (°F)
   6. Average rainfall mm (inches)
   7. Average precipitation days (≥ 0.3 mm)
   8. Average rainy days
   9. Average relative humidity (%) (at 17:30 IST )
  10. Average dew point °C (°F)
  11. Mean monthly sunshine hours
  12. Mean daily sunshine hours
  13. Average ultraviolet index



Boston（16 项）：
   1. Record high °F (°C)
   2. Mean maximum °F (°C)
   3. Mean daily maximum °F (°C)
   4. Daily mean °F (°C)
   5. Mean daily minimum °F (°C)
   6. Mean minimum °F (°C)
   7. Record low °F (°C)
   8. Average precipitation inches (mm)
   9. Average snowfall inches (cm)
  10. Average precipitation days (≥ 0.01 in)
  11. Average snowy days (≥ 0.1 in)
  12. Average relative humidity (%)
  13. Average dew point °F (°C)
  14. Mean monthly sunshine hours
  15. Percentage possible sunshine
  16. Average ultraviolet index


**测试补充**：再验证结果的类型与清洗情况（全为 `str`、无首尾空白、不含 “Climate data for” 标题行与 Source 行），并检查“无气候表页面返回 `None`”。

In [18]:
checks = {}
for name, url in CITY_URLS.items():
    names = list_climate_table_row_names(url)
    checks[name] = names
    assert names is not None
    assert all(isinstance(n, str) for n in names), "列表元素必须都是字符串"
    assert all(n == n.strip() and n for n in names), "不能有首尾空白或空字符串"
    assert not any(re.search(r"climate data for", n, re.I) for n in names), "不能包含标题行"
    assert not any(n.lower().startswith("source") for n in names), "不能包含数据来源行"
    assert not any(n in ("Month", "Jan", "Dec", "Year") for n in names), "不能包含月份列标题"

print("各城市行名数量：")
for name, names in checks.items():
    print(f"  {name:20} {len(names):2} 项 | 前两项：{names[:2]}")

print("\n无气候表页面 ->", list_climate_table_row_names("https://en.wikipedia.org/wiki/Banana"))

# 抽样展示一行完整语义：Harbin 的第一条行名
print("\nHarbin 行名样例：", checks["Harbin"][:3], "...", checks["Harbin"][-1:])
print("\n全部断言通过")

各城市行名数量：
  Harbin               14 项 | 前两项：['Record high °C (°F)', 'Mean maximum °C (°F)']
  Madison, Wisconsin   15 项 | 前两项：['Record high °F (°C)', 'Mean maximum °F (°C)']
  Buenos Aires         11 项 | 前两项：['Record high °C (°F)', 'Mean daily maximum °C (°F)']
  Los Angeles          12 项 | 前两项：['Record high °F (°C)', 'Mean maximum °F (°C)']
  Hyderabad            13 项 | 前两项：['Record high °C (°F)', 'Mean daily maximum °C (°F)']
  Boston               16 项 | 前两项：['Record high °F (°C)', 'Mean maximum °F (°C)']



无气候表页面 -> None

Harbin 行名样例： ['Record high °C (°F)', 'Mean maximum °C (°F)', 'Mean daily maximum °C (°F)'] ... ['Average ultraviolet index']

全部断言通过


## 四、小结

1. **正则函数化**：12 道题都封装成函数并配了多组用例；`fullmatch` 用于“整串校验”，`re.sub` + 捕获组/回调用于替换与重排，`\b` 与前瞻 `(?=…)` 用于精确控制边界。
2. **两个易错点**：空格/下划线互换不能用两次 `re.sub`（会互相撤销），改用回调一次性替换；IP 去前导零必须用 `(?=\d)`，否则 `0.0.0.0` 会变成空段。
3. **网页解析**：`requests` 取页面、`raise_for_status()` 保证 HTTP 错误显式抛出；BeautifulSoup 用 `tbody.find_all("tr", recursive=False)` 拿行，比正则切 HTML 稳定。
4. **结构差异**：同是维基 `wikitable`，标题可能在 `caption` 或首行 `th`，还可能带 `v t e` 前缀（Los Angeles / Hyderabad）；来源行可能用 `td` 或 `th`（Harbin vs Boston），且 `scope="row"` 并非每个页面都有。因此行名判断用了「首格是 `th` + 非纯表头行 + `colspan` ≤ 1 + 非标题行」四条组合规则。
5. **可复现**：全部单元在 `exp03` 环境重启内核后 Run All 通过；访问的 URL 与时间已打印，页面若更新，函数仍按同一套规则工作。